In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
house_candidate = pd.read_csv("house_candidate.csv")
house_state = pd.read_csv("house_state.csv")
governors_county_candidate = pd.read_csv("governors_county_candidate.csv")

FileNotFoundError: [Errno 2] No such file or directory: 'house_candidate.csv'

#### How does House candidate election results in districts with absolute win reflect in presidential elections? 
##### Since house election results are by district, we wanted to see how each district performed, in regards to the margin of victory. 


In [ ]:
county_votes = (house_candidate
                .groupby(["district"])
                .agg(overall_votes=("total_votes", "sum"))
                .reset_index())
sum_values = county_votes.copy() # calculate the total number of votes in each district. 

house_candidate.merge(sum_values, how = 'left', on = 'district') # merge back to the house candidate df
house_candidate = house_candidate.merge(sum_values, how = 'left', on='district') # merge back to the house candidate df

house_candidate['percentage'] = house_candidate['total_votes']/house_candidate['overall_votes'] # calculate percentage of votes for each party. 

In [ ]:
house_candidate = house_candidate[house_candidate['district'] != "United States’s 0th district"]
house_candidate = house_candidate.sort_values(by = "total_votes", ascending=False) # order by the number of votes received by each candidate. 
# only keep top two candidates to calculate the margin of victory. 
candidates = house_candidate.groupby(["district"]).head(2)
# rank by district
candidates.sort_values(by = "district")
candidates = candidates.dropna() # drop NA rows

#### Do the top 5 district political party preferences align with their presidential election outcomes? **What are each party's top 5 supporters?** 




In [ ]:
# for DEM
print(candidates.query("party == 'DEM'").sort_values(by="percentage", ascending=False).head(5)[['district', 'party', 'total_votes','percentage', 'won']])

In [ ]:
print(candidates.query("party == 'REP'").sort_values(by="percentage", ascending=False).head(5)[['district', 'party', 'total_votes','percentage', 'won']])

In [ ]:
print(candidates.query("party == 'IND'").sort_values(by="percentage", ascending=False).head(5)[['district', 'party', 'total_votes','percentage', 'won']])

#### The top 5 districts in support of the Democrats are from Tennessee, New Carolina, Alabama, New York, and Massacheusets. 
#### Take Tennessee's 5th district as an example, would the election results for house candidates at district be indicative of their state election results for president in 2020?

In [ ]:
president_county = pd.read_csv("president_county_candidate.csv")

president_county.head()
sum_prez = president_county.groupby(["county"])['total_votes'].sum()
sum_prez.name = 'sum_votes'
sum_prez
president_county = president_county.merge(sum_prez, how = 'left', on='county') # merge back to the house candidate df
president_county['percentage'] = president_county['total_votes']/president_county['sum_votes'] # calculate percentage of votes for each party. 
president_county

#### Tennessee's 5th district has a 100% vote for Democratic party house candidate in 2020. 
Davidson County, Lewis County, Marshall County, Maury County, Wilson County, and Williamson County are part of the Tennessee's 5th district.
 **For this district, is their politicial preference to a democrat house candidate reflected in their votes for presidential election?**

In [ ]:
tn5th = ['Davidson County', 'Lewis County', 'Marshall County','Maury County', 'Wilson County', 'Williamson County']
tennessee_5th = president_county[president_county['county'].isin(tn5th)]
tennessee_5th = tennessee_5th[tennessee_5th['state'] == 'Tennessee']
tennessee_5th.groupby('party')
tennessee_5th['sum_votes'].sum()
party_votes = tennessee_5th.groupby('party')['total_votes'].sum() # calculate the number of votes each party got, in Tennessee's 5th district. 

plt.bar(party_votes.index, party_votes.values)
plt.xlabel('Party')
plt.ylabel('Total Votes')
plt.title('Total Votes by Party in Tennessee 5th District, 2020 President elections')
plt.show()

Similarly, **Virgina's 9th district** has an absolute win for Republican house candidate in 2020. Bedford County, Bland County, B uchanan County, Carroll County, Craig County, Dickenson County, Floyd County. Franklin County, Giles County, Grayson County, Henry County, Lee County, Montgomery County, Patrick County, Pulaski County, Roanoke County, Russell County, Scott County, Smyth County, Tazewell County, Washington County, Wise County, Wythe County, and some independent cities are in this district.

##### Here, we would impute the presidential election results for this district using the data for all counties. 

In [ ]:
VA9th = ['Bedford County', 'Bland County', 'Buchanan County', 'Carroll County', 'Craig County', 
         'Dickenson County', 'Floyd County', 'Franklin County', 'Giles County', 'Grayson County', 
         'Henry County', 'Lee County', 'Montgomery County', 'Patrick County', 'Pulaski County', 
         'Roanoke County', 'Russell County', 'Scott County', 'Smyth County', 'Tazewell County', 
         'Washington County', 'Wise County', 'Wythe County']
Virginia_9th = president_county[president_county['county'].isin(VA9th)]
Virginia_9th = Virginia_9th[Virginia_9th['state'] == 'Virginia']
Virginia_9th.groupby('party')
Virginia_9th['sum_votes'].sum()
party_votes = Virginia_9th.groupby('party')['total_votes'].sum() # calculate the number of votes each party got, in Virgina's 9th district. 

plt.bar(party_votes.index, party_votes.values)
plt.xlabel('Party')
plt.ylabel('Total Votes')
plt.title('Total Votes by Party in Virgina 9th District, 2020 President elections')
plt.show()

#### **Conclusion**:
Based on the comparison between these results, it seems like the political preference for house candidates does reflect in the presidential election results. Virginia's 9th District has a more stronger prefernece to Republicans than does Tennesse's 5th District to Democrats. 

#### Then, we were curious about the Margin of Victory (MOV) for house candidate elections in each district. 
Did specific party win with absolute competitiveness in the given district? How does the pattern reflect in the presidential elections for this district? 

In [ ]:
# transpose the table into wide format, based on winning situation
mov_candidates = candidates.pivot_table(
    index='district',
    columns='won',
    values='percentage'
)

mov_candidates = mov_candidates.rename(columns={False: 'loser_pct', True: 'winner_pct'})

# filter for absolute win:
abs_win = mov_candidates.loc[mov_candidates['winner_pct']==1]
abs_win['loser_pct']  = 0 # set loser_pct = 0
# merge back to original df, using update
mov_candidates.update(abs_win)

# calculate MOV
mov_candidates['MOV']= mov_candidates['winner_pct'] - mov_candidates['loser_pct']


# competitiveness using pd.cut
mov_candidates['competitiveness'] = pd.cut(
    mov_candidates['MOV'],
    bins=[0, 0.10, 0.20, 1.0],
    labels=['Competitive', 'Likely', 'Safe']
)

# filter for competitive district (top 3)
mov_candidates.query("competitiveness == 'Competitive'").sort_values(by='MOV', ascending=True).head(3)

Iowa's 2nd district has a margin of victory as low as 0.0015%, and California's 25th district with 0.98%.  

In [ ]:
IOWA2nd = [
    "Benton County",
    "Cedar County",
    "Clinton County",
    "Delaware County",
    "Dubuque County",
    "Iowa County",
    "Jackson County",
    "Johnson County",
    "Jones County",
    "Linn County",
    "Muscatine County",
    "Scott County",
    "Washington County"
]
Iowa_2nd = president_county[president_county['county'].isin(IOWA2nd)]
Iowa_2nd = Iowa_2nd[Iowa_2nd['state'] == 'Iowa']
Iowa_2nd.groupby('party')
Iowa_2nd['sum_votes'].sum()
party_votes = Iowa_2nd.groupby('party')['total_votes'].sum() # calculate the number of votes each party got, in Virgina's 9th district. 

plt.bar(party_votes.index, party_votes.values)
plt.xlabel('Party')
plt.ylabel('Total Votes')
plt.title('Total Votes by Party in Iowa 2nd District, 2020 President elections')
plt.show()

What about California's 25th District? 

In [ ]:
california_25th_district_counties = [
    "Los Angeles County",
    "Ventura County"
]

california_25th = president_county[president_county['county'].isin(california_25th_district_counties)]
california_25th = california_25th[california_25th['state'] == 'California']
california_25th.groupby('party')
california_25th['sum_votes'].sum()
party_votes = california_25th.groupby('party')['total_votes'].sum() # calculate the number of votes each party got, in Virgina's 9th district. 

plt.bar(party_votes.index, party_votes.values)
plt.xlabel('Party')
plt.ylabel('Total Votes')
plt.title('Total Votes by Party in California 25th District, 2020 President elections')
plt.show()

**Conclusion**: For our analysis focusing on House Candidates, we can't easily draw a conclusion on how they reflect presidential election results. For house candidates with absolute wins in a certaind districts, people follow a similar trend in presidential elections as well. 
However, for those districts where the MOV for house candidates is really really low, people's vote for presidential election didn't seem to be also scorching. 